### **Orbital Debris Database**   
The Orbital Debris database combines the cleaned UCS and SATCAT datasets into a single analytical source. All de-orbited objects have been removed.   

This notebook builds kinetic_master.csv into a SQLite database using Python, Pandas, and sqlite3, which will be used to run project queries and generate visualizations.   

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
# First we need to load the csv into one massive dataframe. 
df_master = pd.read_csv('../data/clean/kinetic_master.csv', low_memory=False)
df_master

In [ ]:
# Now we have to create a new SQLite database and write the dataframe to it.
# We need to separate the dataframe into multiple tables to avoid redundancy and to make it easier to query later on.
df = df_master.copy()

conn = sqlite3.connect('../data/clean/orbital_debris.db')

df['owner_code'] = df['owner_code'].astype(str).str.strip().str.upper()
df['owner'] = df['owner'].astype(str).str.strip()

owner_name_map = {
    'Spacex': 'SpaceX',
    'spacex': 'SpaceX',
    'spaceX': 'SpaceX',
    'Swarm Technologies': 'SpaceX',
    'Space Exploration Technologies Corp.': 'SpaceX'
}

# Patch: propagate UCS flags to owner level (any-true logic)
flag_cols = ['is_commercial', 'is_government', 'is_military', 'is_civil']
owner_flags = df.groupby('owner_code')[flag_cols].max().reset_index()

# Overwrite the per-row flags with the aggregated owner-level flags
df = df.drop(columns=flag_cols).merge(owner_flags, on='owner_code', how='left')

df['owner'] = df['owner'].replace(owner_name_map)

# keep orbit_type stable
df['orbit_type'] = df['orbit_type'].fillna('Other/Misc')

# launch_id is synthetic: derive from COSPAR prefix YYYY-NNN
df['launch_id'] = df['cospar_id'].astype(str).str.extract(r'^(\d{4}-\d{3})', expand=False)
df['launch_id'] = df['launch_id'].fillna('UNKNOWN')

df_ownership_operators = df[
    ['owner_code', 'owner', 'country_operator', 'users',
     'is_commercial', 'is_government', 'is_military', 'is_civil',
     'contractor', 'contractor_country']
].drop_duplicates(subset=['owner_code'])

df_launch_events = df[
    ['launch_id', 'launch_date', 'launch_year', 'launch_site']
].drop_duplicates(subset=['launch_id'])

df_satellites = df[
    ['norad_id', 'cospar_id', 'object_name', 'satellite_name', 'official_name',
     'object_type', 'category', 'ops_status', 'data_status', 'in_orbit',
     'owner_code', 'launch_id']
].drop_duplicates(subset=['norad_id'])

df_orbital_data = df[
    ['norad_id', 'orbit_class', 'orbit_type', 'period_minutes', 'perigee_km',
     'apogee_km', 'inclination_degrees', 'eccentricity', 'semi_major_axis_km',
     'launch_mass_kg', 'proxy_mass_kg', 'dry_mass_kg', 'power_watts',
     'proxy_power_watts', 'rcs', 'rcs_class']
].drop_duplicates(subset=['norad_id'])

df_ucs_details = df[
    ['norad_id', 'lifetime_years', 'sat_age_years',
     'primary_purpose', 'detailed_purpose', 'geo_longitude', 'un_registry']
].drop_duplicates(subset=['norad_id'])

df_risk_assessment = df[
    ['norad_id', 'velocity_kms', 'kinetic_joules', 'is_zombie']
].drop_duplicates(subset=['norad_id'])

# Write each dataframe to SQLite using schema-aligned table names
df_satellites.to_sql('satellites', conn, if_exists='replace', index=False)
df_orbital_data.to_sql('orbital_data', conn, if_exists='replace', index=False)
df_ucs_details.to_sql('ucs_details', conn, if_exists='replace', index=False)
df_risk_assessment.to_sql('risk_assessment', conn, if_exists='replace', index=False)
df_ownership_operators.to_sql('ownership_operators', conn, if_exists='replace', index=False)
df_launch_events.to_sql('launch_events', conn, if_exists='replace', index=False)

# Quick sanity checks (row counts + duplicate key audit)
checks = [
    # TABLE_NAME,           KEY_COLUMN
    ('satellites',          'norad_id'),
    ('orbital_data',        'norad_id'),
    ('ucs_details',         'norad_id'),
    ('risk_assessment',     'norad_id'),
    ('ownership_operators', 'owner_code'),
    ('launch_events',       'launch_id')
]

for table_name, key_col in checks:
    row_count = pd.read_sql(
        f"SELECT COUNT(*) AS count FROM {table_name}",
        conn).iloc[0]['count']
    
    dup_count = pd.read_sql(
        f"SELECT COUNT(*) AS count FROM (SELECT {key_col} FROM {table_name} GROUP BY {key_col} HAVING COUNT(*) > 1)",
        conn).iloc[0]['count']
        
    print(f"{table_name:20} rows={int(row_count):,} | duplicate_{key_col}={int(dup_count):,}")

conn.commit()
conn.close()

print('SQLite build complete: ../data/clean/orbital_debris.db')